# ByteEmbed — FINAL retrieval study (BGE-M3 teacher, verified language set)

**The one notebook for all the finalized experiments** — smoke → main 6-model grid → both boundary
arms × all sizes (6 more) → teacher baseline → AfriQA table. **12 trained students total**, all
session-partitionable: every training cell has `SESSION_MODELS`/`ARM_MODELS` + `MAX_CONCURRENT`
knobs, so you can split the work across several Colab sessions sharing one Drive folder (disjoint
model lists never collide). The same 12 jobs dispatch on a SLURM cluster via
`bash slurm/submit_all.sh` (identical part-file convention — Colab and SLURM are interchangeable).

**Design (locked — see `RETRIEVAL_EXPERIMENT.md`):**
- **Languages (10):** te, bn, sw, yo, am, ha, rw (lower-resource; bn = Joshi class 3, the one stated
  relaxation) + en, zh, ar (anchors). All 10 trained (~42k sentences each).
- **Models (12):** byt5 vs mt5 × {small, base, large} (main grid, 6) + boundary arms `teacher` and
  `random` × byte-{small, base, large} (6).
- **Teacher:** **BGE-M3** (`BAAI/bge-m3`, retrieval-trained, 1024-d), targets cached once
- **Objective (retrieval-only):** pure InfoNCE (τ=0.05, queue 8192), AdamW lr 2e-4, **batch 64,
  100k steps for every model (iso-step)**, `attn` pooling
- **Eval:** shallow = **Belebele only** (all 10); deep = **ONE benchmark per language** — MIRACL dev
  (te/bn/sw/yo/en/zh/ar) · Amharic-PR (am) · CIRAL Test A (ha, cross-lingual, flagged) · AfriQA
  (rw, cross-lingual REVERSE, flagged). Full stats everywhere: nDCG@10, P@10, R@10, R@100, MRR@10.
- **Baseline:** **the teacher only** (BGE-M3 on the identical battery = the ceiling).
- **Multi-session rule:** start ONE session first until its log prints `reusing cached targets`,
  then start the rest; keep `RUN_BASELINES=True` in exactly one session.
- **Caveat to report:** yo is not in XLM-R/CC-100 (the teacher's backbone) — both students inherit
  the same weakened targets, so the comparison stays fair.

Everything is resumable; smoke first. First eval streams the CIRAL-ha corpus once (the big one-time
download) — pools cache to `checkpoints/` and every later model reuses them.

### 1. GPU check — confirm you're on an A100 (Runtime → Change runtime type → A100)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

### 2. Clone repo + install deps
Imports work from the repo root even if the editable install is skipped.

In [ ]:
import os
os.chdir('/content')
REPO = 'https://github.com/Aarushvinod/embedding-research.git'
if not os.path.isdir('/content/embedding-research'):
    !git clone -q $REPO
os.chdir('/content/embedding-research')
!git pull -q
!pip install -q -r requirements-cloud.txt
!pip install -q -e . || echo '(editable install skipped — running from repo root is fine)'
print('setup done | cwd', os.getcwd())

### 3. Teacher check + persist to Drive
BGE-M3 loads via sentence-transformers (already a dep) — no extra install, no fairseq2. Point
`PERSIST` at the **same** `byteembed_lowres` folder as the SONAR runs: the balanced-data cache is
reused; the BGE-M3 teacher targets get their own cache file (`teachertargets_bge-m3_*`), so nothing
collides. Skip the Drive block to run on ephemeral disk.

In [ ]:
from huggingface_hub import hf_hub_download
_ = hf_hub_download('BAAI/bge-m3', 'config.json')   # reachable? (weights download on first teacher load)
print('BGE-M3 reachable — teacher will be BAAI/bge-m3 (retrieval-trained, 1024-d)')

from google.colab import drive
drive.mount('/content/drive')
import os, shutil
PERSIST = '/content/drive/MyDrive/byteembed_lowres'   # SAME folder as the SONAR runs -> shared data cache
for d in ('results', 'checkpoints'):
    os.makedirs(f'{PERSIST}/{d}', exist_ok=True)
    if not os.path.islink(d):
        if os.path.isdir(d): shutil.rmtree(d)
        os.symlink(f'{PERSIST}/{d}', d)
print('persisting results/ and checkpoints/ to', PERSIST)

### 4. Smoke test (~5 min) — validate the pipeline with the NEW teacher
3 langs (am/sw/en), 2 tiny students, tiny eval. Confirms: BGE-M3 loads → targets precompute + cache →
train → full eval battery (incl. QA-retrieval) → save. **Check the log says 'BGE-M3 ... loaded' — not
SONAR, not LaBSE.**

In [ ]:
from byte_embed.run_lowresource import run
_ = run(smoke=True, teacher_name='bge-m3', pooling='attn', out='results/retrieval_bgem3_smoke.json')

### 5. Main grid — 6 students, session-configurable
**100k steps × batch 64 for every model (iso-step), `attn` pooling.** Edit the `SESSION_*` knobs to
control exactly what THIS Colab session trains and how many run concurrently — so you can split the
grid across several sessions sharing the same Drive folder (disjoint model lists never collide:
part-files and checkpoints are per-model).

**Multi-session recipe:** start session 1 alone until its log prints `reusing cached targets`
(the one-time BGE-M3 pass), then start the others. Example split — session 1:
`['byte-large','subword-large']` (the slow pair) · session 2: the four small/base models ·
session 3: the boundary arms (step 5b). Leave `RUN_BASELINES=True` in exactly ONE session.
On a SLURM cluster, `bash slurm/submit_all.sh` dispatches the same 12 trainings as dependency-chained
jobs (same part-file convention — Colab and SLURM are interchangeable mid-study).

In [ ]:
# ===== SESSION CONFIG — edit these three lines per Colab session =====
SESSION_MODELS = None    # None = all 6; or a subset, e.g. ['byte-large', 'subword-large']
MAX_CONCURRENT = 3       # concurrent trainings on THIS gpu (A100-80GB: 3; G4-96GB: 4-5)
RUN_BASELINES  = True    # keep True in exactly ONE session (the one finishing last)

from byte_embed.run_parallel import parallel
parallel(
    out='results/retrieval_bgem3.json',
    teacher_name='bge-m3',            # THE one change vs the SONAR study
    pooling='attn',                   # same pooling for byte AND subword (fair)
    steps=100000,                     # the CAP — patience below can stop earlier
    patience=10,                      # stop after 10 x 1000-step windows w/o loss improving > 1e-3;
                                      #   identical setting for every model; realized steps land in
                                      #   results as steps_run — REPORT them per model
    only=SESSION_MODELS,
    max_concurrent=MAX_CONCURRENT,
    with_baselines=RUN_BASELINES,
)
# Valid names: byte-small, subword-small, byte-base, subword-base, byte-large, subword-large.
# Resumable — re-running skips finished models; a disconnect loses at most 5k steps (ckpt_every).

### 5b. Boundary-injection arms — BOTH arms × ALL 3 byte sizes (6 models)
Arm **B** (`teacher`): markers inserted where BGE-M3's tokenizer would split — segmentation info,
zero vocab table. Arm **C** (`random`): the placebo — same per-sentence marker count at random
character positions. Teacher targets stay clean; each arm trains AND evals with its own transform,
in its own results file and `_b-{arm}` checkpoint namespace — so the arms collide with nothing and
can run in a **separate Colab session** alongside the main grid (same sequencing rule: wait for
`reusing cached targets`). Use the knobs to split further, e.g. one session per arm, or
`ARM_MODELS=['byte-large']` sessions. Interpreting: B > C ≈ A → segmentation info helps;
B ≈ C > A → artifact, no credit to the tokenizer; B ≈ C ≈ A → byte needs nothing from segmentation.

In [ ]:
# ===== BOUNDARY-ARM SESSION CONFIG =====
ARMS           = ('teacher', 'random')   # both arms; or ('teacher',) / ('random',) to split sessions
ARM_MODELS     = None                    # None = byte-small/base/large; or e.g. ['byte-small']
MAX_CONCURRENT = 3

from byte_embed.run_parallel import parallel
for arm in ARMS:
    parallel(
        out=f'results/retrieval_bgem3_b{arm}.json',
        teacher_name='bge-m3', pooling='attn',
        steps=100000, patience=10,       # same cap + plateau stop as the main grid (fair)
        boundary=arm, only=ARM_MODELS, max_concurrent=MAX_CONCURRENT,
        with_baselines=False,            # the teacher baseline lives in the MAIN results file only
    )

# Three-arm comparison — A (main run) vs B (teacher-boundaries) vs C (random) per size:
import json
def _grab(path, name):
    try:
        r = json.load(open(path))['models'][name]
        qa = r.get('qa_retrieval') or {}
        return {'steps_run': r.get('steps_run'),
                'Belebele': (r.get('means') or {}).get('belebele_ndcg@10'),
                'MIRACL': (r.get('miracl') or {}).get('ndcg@10_mean'),
                'AmharicPR': (qa.get('amharicpr') or {}).get('ndcg@10_mean'),
                'CIRAL_ha': ((qa.get('ciral') or {}).get('per_lang', {}).get('ha') or {}).get('ndcg@10'),
                'AfriQA_rw': ((qa.get('afriqa') or {}).get('per_lang', {}).get('rw') or {}).get('ndcg@10')}
    except Exception as e:
        return f'(pending: {type(e).__name__})'
for size in ('small', 'base', 'large'):
    print(f"--- byte-{size} ---")
    for arm, path in [('A raw   ', 'results/retrieval_bgem3.json'),
                      ('B teach ', 'results/retrieval_bgem3_bteacher.json'),
                      ('C random', 'results/retrieval_bgem3_brandom.json')]:
        print(f'  {arm}', _grab(path, f'byte-{size}'))

### 6. Teacher baseline + summary (run ONCE, after all sessions finish)
Scores **BGE-M3 itself** on the identical battery (the ceiling the students chase — the only baseline
we measure), merges any remaining part-files, and prints the full table. If `RUN_BASELINES=True`
already ran in a session, this is just a re-print. Note: any model still missing from the results
would start training *sequentially* here — so run this after the training sessions are done.

In [ ]:
from byte_embed.run_lowresource import run
_ = run(out='results/retrieval_bgem3.json', teacher_name='bge-m3', pooling='attn', steps=50000)

### 6b. AfriQA table (rw's deep benchmark + the ha/sw/yo reverse-axis probe)
AfriQA now runs **in the main eval** (default battery): **rw is its headline** — Kinyarwanda's deep
benchmark (347 native questions → English gold passages + 20k English distractors; flagged
cross-lingual-REVERSE, the mirror of ha's CIRAL standard) — with ha/sw/yo riding along as the
reverse-axis probe. This cell backfills any models evaluated before AfriQA joined the battery
(additive — existing metrics kept) and prints the per-language table.

In [ ]:
# AfriQA backfill (no-op if the main run already computed it) + per-language table.
from byte_embed.reeval import reeval
reeval('results/retrieval_bgem3.json', pooling='attn', qa_only=True, benchmarks=('afriqa',))

# African question -> English passage; rw is the headline (its deep benchmark), ha/sw/yo = probe:
import json
res = json.load(open('results/retrieval_bgem3.json'))
print(f"{'model':16}" + "".join(f"{l:>10}" for l in ('rw', 'ha', 'sw', 'yo')))
for name, r in res['models'].items():
    per = ((r.get('qa_retrieval') or {}).get('afriqa') or {}).get('per_lang') or {}
    row = "".join(f"{(per.get(l) or {}).get('ndcg@10', float('nan')):>10.3f}" if per.get(l) else f"{'-':>10}"
                  for l in ('rw', 'ha', 'sw', 'yo'))
    print(f"{name:16}{row}")

### 7. Download results
Already on Drive if you ran the persist cell; otherwise grab the JSON here.

In [ ]:
from google.colab import files
files.download('results/retrieval_bgem3.json')
for f in ('results/retrieval_bgem3_bteacher.json', 'results/retrieval_bgem3_brandom.json'):
    try: files.download(f)
    except Exception as e: print('skip', f, '-', e)